# LinkedIn Jobs Data Pipeline
## Inspect, Load, and Profile LinkedIn Job Postings

**Thesis:** Curriculum–Industry Alignment in Armenian IT Education  
**Source:** LinkedIn Armenia (Apify scrape, 2026-03-20)  
**Notebook purpose:** Load the LinkedIn jobs dataset, inspect its structure,  
identify useful fields for NLP skill extraction, and produce a standardized subset.  
**Output:** `data/processed/jobs/linkedin_jobs_standardized.csv`

---

## Setup

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path('../../')
JOBS_CSV     = PROJECT_ROOT / 'data/raw/jobs/linkedin_jobs_raw.csv'
OUT_DIR      = PROJECT_ROOT / 'data/processed/jobs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)

print('Ready. LinkedIn raw jobs file:', JOBS_CSV.resolve())

---
## Step 1 — Load the Dataset

In [ ]:
df = pd.read_csv(JOBS_CSV, low_memory=False)
print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} job postings')
print(f'Columns: {df.shape[1]}')

---
## Step 2 — Inspect All Columns

In [ ]:
# Show all columns, non-null counts, and dtypes
print('Column inventory:')
info = pd.DataFrame({
    'column': df.columns,
    'non_null': df.notna().sum().values,
    'pct_filled': (df.notna().mean() * 100).round(1).values,
    'dtype': df.dtypes.values
})
info = info.sort_values('pct_filled', ascending=False).reset_index(drop=True)
info

In [ ]:
# Show a few sample rows (key fields only)
sample_cols = ['title', 'companyName', 'location', 'employmentType',
               'seniorityLevel', 'postedAt', 'descriptionText']
available   = [c for c in sample_cols if c in df.columns]
df[available].head(10)

---
## Step 3 — Profile Key Fields

Understanding the distribution of values helps plan the NLP pipeline.

In [ ]:
# Job titles — top 30
if 'title' in df.columns:
    print('Top 30 job titles:')
    print(df['title'].value_counts().head(30).to_string())

In [ ]:
# Seniority levels
if 'seniorityLevel' in df.columns:
    print('Seniority levels:')
    print(df['seniorityLevel'].value_counts())

In [ ]:
# Employment types
if 'employmentType' in df.columns:
    print('Employment types:')
    print(df['employmentType'].value_counts())

In [ ]:
# Job functions
if 'jobFunction' in df.columns:
    print('Job functions:')
    print(df['jobFunction'].value_counts().head(20))

In [ ]:
# Industries
if 'industries' in df.columns:
    print('Industries:')
    print(df['industries'].value_counts().head(20))

In [ ]:
# Top companies
if 'companyName' in df.columns:
    print('Top 20 companies by job count:')
    print(df['companyName'].value_counts().head(20).to_string())

In [ ]:
# Description availability and average length
if 'descriptionText' in df.columns:
    desc = df['descriptionText'].dropna()
    desc_len = desc.str.len()
    print(f'Jobs with text description: {len(desc):,} / {len(df):,}')
    print(f'Avg description length: {desc_len.mean():.0f} chars')
    print(f'Min: {desc_len.min()}, Max: {desc_len.max()}')
    print()
    print('Sample description (first job):')
    print(desc.iloc[0][:1000])

---
## Step 4 — Identify Fields Useful for NLP

For skill extraction, the following fields are most valuable:

| Field | Use |
|---|---|
| `title` | Identify job role categories |
| `descriptionText` | **Primary NLP target** — extract required skills |
| `seniorityLevel` | Stratify results by experience level |
| `employmentType` | Filter if needed (e.g., focus on full-time) |
| `jobFunction` | Group jobs by domain |
| `industries` | Segment by industry |
| `companyName` | Group by employer |
| `standardizedTitle` | Normalized title for clustering |
| `postedAt` | Temporal analysis (optional) |

In [ ]:
# Verify standardizedTitle availability
if 'standardizedTitle' in df.columns:
    print('standardizedTitle — top 20:')
    print(df['standardizedTitle'].value_counts().head(20))
else:
    print('standardizedTitle column not present')

In [ ]:
# Check for duplicates by job ID
if 'id' in df.columns:
    n_dupes = df.duplicated(subset='id').sum()
    print(f'Duplicate job IDs: {n_dupes}')
else:
    n_dupes = df.duplicated().sum()
    print(f'Fully duplicate rows: {n_dupes}')

---
## Step 5 — Save Cleaned Subset

Select and save the fields most relevant to the thesis analysis.

In [ ]:
# Select key columns that exist in this dataset
desired_cols = [
    'id', 'title', 'standardizedTitle', 'companyName', 'location',
    'employmentType', 'seniorityLevel', 'jobFunction', 'industries',
    'postedAt', 'descriptionText', 'link', 'applyUrl'
]
keep_cols = [c for c in desired_cols if c in df.columns]
print(f'Keeping {len(keep_cols)} columns: {keep_cols}')

jobs_clean = df[keep_cols].copy()

# Drop duplicates if id column is present
if 'id' in jobs_clean.columns:
    before = len(jobs_clean)
    jobs_clean = jobs_clean.drop_duplicates(subset='id')
    print(f'Dropped {before - len(jobs_clean)} duplicate IDs')

print(f'Final cleaned dataset shape: {jobs_clean.shape}')
jobs_clean.head(3)

In [ ]:
# Save standardized LinkedIn jobs subset
out_path = OUT_DIR / 'linkedin_jobs_standardized.csv'
jobs_clean.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Saved: {out_path}  ({len(jobs_clean)} rows)')

---
## Summary

| Metric | Value |
|---|---|
| Total job postings | 992 |
| Source | LinkedIn (Apify scrape) |
| Country | Armenia (AM) — 100% |
| Scrape date | 2026-03-20 |
| Output file | `data/processed/jobs/linkedin_jobs_standardized.csv` |

**Key NLP field:** `descriptionText` — the primary source for skill extraction.

### Next Steps
1. See `02_staffam_scraping.ipynb` for Staff.am IT jobs scraping
2. After both sources are collected: merge into `final_jobs_dataset.csv`
3. Begin skill extraction in Phase 3
